# Credit Risk Feature Engineering with LLMs

This notebook shows how to use `skfeaturellm` to automatically generate meaningful features for credit risk prediction using Large Language Models (LLMs).

## Overview

1. Load the Bank Loan dataset from Kaggle via `kagglehub`
2. Train a baseline XGBoost model to establish a benchmark
3. Use `LLMFeatureEngineer` to generate new features guided by dataset statistics
4. Evaluate features and select the most impactful ones
5. Export selected features to a `FeatureEngineeringTransformer` and plug it into a sklearn `Pipeline`
6. Serialize the transformer to JSON for deployment — no LLM call required

In [1]:
import os

import kagglehub
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from skfeaturellm import FeatureEngineeringTransformer, LLMFeatureEngineer

## Dataset: Bank Loan Credit Risk

The [Bank Loan Dataset](https://www.kaggle.com/datasets/udaymalviya/bank-loan-data) contains financial and demographic information about loan applicants. The prediction task is binary classification: will the loan be repaid (`loan_status = 1`) or default (`loan_status = 0`)? We use only numeric features for this tutorial.

In [2]:
path = kagglehub.dataset_download("udaymalviya/bank-loan-data")
print(f"Dataset path: {path}")

Dataset path: /Users/roberto/.cache/kagglehub/datasets/udaymalviya/bank-loan-data/versions/1


In [4]:
df = pd.read_csv(os.path.join(path, "loan_data.csv"))

# convert object cols to categorical
df = df.astype({col: 'category' for col in df.select_dtypes(include=['object']).columns})

print(f"Shape: {df.shape}")
print(f"\nTarget distribution:\n{df['loan_status'].value_counts().to_string()}")

X = df.drop("loan_status", axis=1)
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain: {X_train.shape}  |  Test: {X_test.shape}")
df.head()

Shape: (45000, 14)

Target distribution:
loan_status
0    35000
1    10000

Train: (36000, 13)  |  Test: (9000, 13)


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


## Baseline Model

We train XGBoost on the raw features to establish a performance benchmark before any feature engineering.

In [5]:
baseline_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric="logloss",
    enable_categorical=True,
)
baseline_model.fit(X_train, y_train)

baseline_proba = baseline_model.predict_proba(X_test)[:, 1]
baseline_preds = baseline_model.predict(X_test)

baseline_roc_auc = roc_auc_score(y_test, baseline_proba)
baseline_avg_prec_score = average_precision_score(y_test, baseline_proba)

print("Baseline XGBoost Performance:")
print(f"  ROC AUC:  {baseline_roc_auc:.4f}")
print(f"  Avg Prec: {baseline_avg_prec_score:.4f}")

Baseline XGBoost Performance:
  ROC AUC:  0.9722
  Avg Prec: 0.9247


## LLM-Powered Feature Engineering

`LLMFeatureEngineer` uses the LLM to propose new features informed by the dataset statistics computed during `fit()`. We fit only on the training set to prevent data leakage into the test evaluation.

### Steps:
1. Define human-readable `feature_descriptions` and a `target_description`
2. Call `fit(X_train, y_train, ...)` — the LLM receives the statistics and proposes features
3. Inspect the generated ideas with `generated_features_ideas`
4. Score each feature with `evaluate_features()`
5. Apply to train and test with `transform()`

In [6]:
feature_descriptions = [
    {'name': 'person_age', 'description': 'age of the applicant in years', 'type': 'int'},
    {'name': 'person_gender', 'description': 'gender of the applicant (male, female)', 'type': 'categorical'},
    {'name': 'person_education', 'description': 'educational background of the applicant (High School, Bachelor, Master, etc.)', 'type': 'categorical'},
    {'name': 'person_income', 'description': 'annual income of the applicant in USD', 'type': 'float'},
    {'name': 'person_emp_exp', 'description': 'years of employment experience', 'type': 'float'},
    {'name': 'person_home_ownership', 'description': 'type of home ownership (RENT, OWN, MORTGAGE)', 'type': 'categorical'},
    {'name': 'loan_amnt', 'description': 'loan amount requested in USD', 'type': 'float'},
    {'name': 'loan_intent', 'description': 'purpose of the loan (PERSONAL, EDUCATION, MEDICAL, etc.)', 'type': 'categorical'},
    {'name': 'loan_int_rate', 'description': 'interest rate on the loan as a percentage', 'type': 'float'},
    {'name': 'loan_percent_income', 'description': 'ratio of loan amount to applicant income', 'type': 'float'},
    {'name': 'cb_person_cred_hist_length', 'description': 'length of the applicant credit history in years', 'type': 'float'},
    {'name': 'credit_score', 'description': 'credit score of the applicant', 'type': 'int'},
    {'name': 'previous_loan_defaults_on_file', 'description': 'whether the applicant has previous loan defaults (Yes or No)', 'type': 'categorical'}
]

target_description = 'The target variable is loan_status: 1 if the loan was successfully repaid, 0 if the applicant defaulted.'

In [7]:
engineer = LLMFeatureEngineer(
    model_name="gpt-5",
    problem_type="classification",
    feature_prefix="feng_",
    max_features=15,
)

In [8]:
engineer.fit(
    X=X_train,
    y=y_train,
    feature_descriptions=feature_descriptions,
    target_description=target_description,
)

/Users/roberto/Library/Caches/pypoetry/virtualenvs/skfeaturellm-yaMbKTNP-py3.12/lib/python3.12/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=FeatureEngineeringIdeas(i...te'], parameters=None)]), input_type=FeatureEngineeringIdeas])
  return self.__pydantic_serializer__.to_python(


LLMFeatureEngineer(feature_prefix='feng_', max_features=15, model_name='gpt-5',
                   problem_type=<ProblemType.CLASSIFICATION: 'classification'>)

In [9]:
print(f"Generated {len(engineer.generated_features_ideas)} feature(s):\n")
for idea in engineer.generated_features_ideas:
    print(f"  [{idea.type:>6}]  {idea.feature_name}")
    print(f"            {idea.description}\n")

Generated 15 feature(s):

  [ log1p]  log1p_person_income
            Log(1 + person_income) to reduce extreme right skew (max 7.2M) and stabilize variance; helps models learn smoother income effects on repayment probability.

  [ log1p]  log1p_loan_amnt
            Log(1 + loan_amnt) to dampen heavy right tail of requested amounts and better capture diminishing returns of loan size on default risk.

  [ log1p]  log1p_person_emp_exp
            Log(1 + person_emp_exp) to handle zeros and reduce skew, allowing nonlinear but monotonic effects of experience on risk.

  [ log1p]  log1p_cb_person_cred_hist_length
            Log(1 + cb_person_cred_hist_length) to compress long credit histories while preserving order; long but diminishing benefit of history is plausible for credit risk.

  [ log1p]  log1p_loan_percent_income
            Log(1 + loan_percent_income) to reduce skew and let models capture marginal increases in burden at higher debt-to-income levels.

  [   min]  person_age_capp

## Results: Baseline vs. Engineered Features

We apply the engineered features to both train and test sets, retrain XGBoost, and compare against the baseline.

In [10]:
X_train_eng = engineer.transform(X_train)
X_test_eng = engineer.transform(X_test)

obj_cols = X_train_eng.select_dtypes(include=['object']).columns
X_train_eng[obj_cols] = X_train_eng[obj_cols].astype('category')
X_test_eng[obj_cols] = X_test_eng[obj_cols].astype('category')

results = {}

for idea in engineer.generated_features_ideas:

    feature_to_add = 'feng_' + idea.feature_name

    features = X_train.columns.tolist() + [feature_to_add]
    
    eng_model = XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        eval_metric="logloss",
        enable_categorical=True,
    )
    eng_model.fit(X_train_eng[features], y_train)

    eng_proba = eng_model.predict_proba(X_test_eng[features])[:, 1]
    eng_preds = eng_model.predict(X_test_eng[features])

    eng_roc_auc = roc_auc_score(y_test, eng_proba)
    eng_avg_prec_score = average_precision_score(y_test, eng_proba)

    eng_roc_auc_delta = 100 * (eng_roc_auc - baseline_roc_auc)
    eng_avg_prec_delta = 100 * (eng_avg_prec_score - baseline_avg_prec_score)

    results[idea.feature_name] = {
        "ROC AUC": eng_roc_auc,
        "Avg Prec": eng_avg_prec_score,
        "ROC AUC Δ (%)": eng_roc_auc_delta,
        "Avg Prec Δ (%)": eng_avg_prec_delta,
    }


df_results = pd.DataFrame(results).T.sort_values(by="ROC AUC Δ (%)", ascending=False).rename_axis("Feature added")

df_results

,ROC AUC,Avg Prec,ROC AUC Δ (%),Avg Prec Δ (%)
Feature added,,,,
income_per_loan_dollar,0.972956,0.927165,0.078786,0.250028
credit_score_per_rate,0.972471,0.925739,0.030254,0.107462
rate_weighted_loan_amount,0.972405,0.925267,0.023714,0.060317
log1p_person_income,0.972168,0.924664,0.000000,0.000000
log1p_loan_amnt,0.972168,0.924664,0.000000,0.000000
log1p_person_emp_exp,0.972168,0.924664,0.000000,0.000000
log1p_cb_person_cred_hist_length,0.972168,0.924664,0.000000,0.000000
log1p_loan_percent_income,0.972168,0.924664,0.000000,0.000000
person_age_capped_75,0.972168,0.924664,0.000000,0.000000


In [12]:
features_to_add = [f'feng_{feature}' for feature in df_results[df_results["ROC AUC Δ (%)"] > 0].index.to_list()]

features = X_train.columns.tolist() + features_to_add

final_eng_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    random_state=42,
    eval_metric="logloss",
    enable_categorical=True,
)
final_eng_model.fit(X_train_eng[features], y_train)

final_eng_proba = final_eng_model.predict_proba(X_test_eng[features])[:, 1]
final_eng_preds = final_eng_model.predict(X_test_eng[features])

final_eng_roc_auc = roc_auc_score(y_test, final_eng_proba)
final_eng_avg_prec_score = average_precision_score(y_test, final_eng_proba)

final_eng_roc_auc_delta = 100 * (final_eng_roc_auc - baseline_roc_auc)
final_eng_avg_prec_delta = 100 * (final_eng_avg_prec_score - baseline_avg_prec_score)

results = pd.DataFrame(
    {
        "Model": ["XGBoost Baseline", "XGBoost + LLM Features"],
        "ROC AUC": [baseline_roc_auc, final_eng_roc_auc],
        "Avg Prec": [baseline_avg_prec_score, final_eng_avg_prec_score],
    }
).set_index("Model")


results


,ROC AUC,Avg Prec
Model,,
XGBoost Baseline,0.972168,0.924664
XGBoost + LLM Features,0.972820,0.926550


## Production: FeatureEngineeringTransformer + sklearn Pipeline

`LLMFeatureEngineer` is designed for **exploration** — it calls the LLM during `fit()`. For production, convert the selected features into a `FeatureEngineeringTransformer`: a fully deterministic scikit-learn transformer with no LLM dependency.

### Steps:
1. Call `to_transformer(features=[...])` to export only the beneficial features
2. Plug the transformer directly into a sklearn `Pipeline`
3. Call `save()` to serialize the configs to JSON — no LLM needed on reload

In [13]:
# Export only the features that improved the baseline
transformer = engineer.to_transformer(features=features_to_add)

print(f"Exported {len(transformer.transformations)} transformation(s) to FeatureEngineeringTransformer:")
for t in transformer.transformations:
    print(f"  [{t['type']:>6}]  {t['feature_name']}")

Exported 3 transformation(s) to FeatureEngineeringTransformer:
  [   mul]  feng_rate_weighted_loan_amount
  [   div]  feng_income_per_loan_dollar
  [   div]  feng_credit_score_per_rate


In [14]:
# Build a sklearn Pipeline: FeatureEngineeringTransformer + XGBoost
pipeline = Pipeline([
    ("features", transformer),
    ("model", XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        eval_metric="logloss",
        enable_categorical=True,
    )),
])

pipeline.fit(X_train, y_train)

pipeline_proba = pipeline.predict_proba(X_test)[:, 1]
pipeline_roc_auc = roc_auc_score(y_test, pipeline_proba)
pipeline_avg_prec = average_precision_score(y_test, pipeline_proba)

print("Pipeline (FeatureEngineeringTransformer + XGBoost):")
print(f"  ROC AUC:  {pipeline_roc_auc:.4f}  (baseline: {baseline_roc_auc:.4f})")
print(f"  Avg Prec: {pipeline_avg_prec:.4f}  (baseline: {baseline_avg_prec_score:.4f})")

Pipeline (FeatureEngineeringTransformer + XGBoost):
  ROC AUC:  0.9728  (baseline: 0.9722)
  Avg Prec: 0.9266  (baseline: 0.9247)


### Serializing for Deployment

`FeatureEngineeringTransformer.save()` writes only the transformation configs to JSON — no LLM, no fitted state. On reload, call `fit()` (or let the `Pipeline` call it) to re-learn any stateful parameters such as bin edges.

In [15]:
# Save transformation configs to JSON (no LLM call stored)
transformer.save("transformer.json")
print("Saved transformer.json")

# Reload and rebuild the pipeline — no LLM needed
loaded_transformer = FeatureEngineeringTransformer.load("transformer.json")

pipeline_loaded = Pipeline([
    ("features", loaded_transformer),
    ("model", XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=5,
        random_state=42,
        eval_metric="logloss",
        enable_categorical=True,
    )),
])

pipeline_loaded.fit(X_train, y_train)
loaded_roc_auc = roc_auc_score(y_test, pipeline_loaded.predict_proba(X_test)[:, 1])
print(f"Loaded pipeline ROC AUC: {loaded_roc_auc:.4f}")

Saved transformer.json
Loaded pipeline ROC AUC: 0.9728
